In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd

from infer_subc.core.file_io import (list_image_files,
                                     read_czi_image,
                                     export_inferred_organelle)
from infer_subc.core.img import label

import napari

viewer=napari.Viewer()

File Paths & User Input

In [ ]:
in_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/segmented"
out_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/segmented/out"
log_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/segmented/out/csv"
log_name="log"
suffix = "-cell"
file_type=".tiff"

Batch Process (Do not edit any code below)

In [ ]:
in_path = Path(in_path)
out_path = Path(out_path)
img_file_list = [im for im in list_image_files(in_path, file_type) if str(im.stem)[-len(suffix):]==suffix]
log_entries = []
for img_f in img_file_list:
    img_data, meta_dict = read_czi_image(img_f)
    img_out = label(img_data).copy()
    img_out[img_out!=np.bincount(np.ravel(img_out)[np.ravel(img_out)!= 0]).argmax()] = 0
    img_out[img_out>0] = 1
    input_voxels = np.sum(img_data>0)
    cleaned_voxels = np.sum(img_out>0)
    export_inferred_organelle(img_out, "cleaned", meta_dict, out_path)
    print(f"✅ Saved to: {img_f.name} — cleaned voxels: {cleaned_voxels}")
    log_entries.append({
            "file": img_f.name,
            "input_voxels": int(input_voxels),
            "cleaned_voxels": int(cleaned_voxels),
            "removed_voxels": int(input_voxels - cleaned_voxels),
            "percent_retained": round(100 * cleaned_voxels / input_voxels, 2) if input_voxels > 0 else 0.0
    })
    viewer.add_labels(img_data, name=f"Raw: {img_f.stem}")
    viewer.add_labels(img_out, name=f"Cleaned: {img_f.stem}")

log_df = pd.DataFrame(log_entries)
log_df.to_csv(log_path+'/'+log_name+'.csv', index=False)
print(f"📝 Log saved to: {log_path}")